In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations, chain


df_CP = pd.read_csv("./rag/crows-pairs/data/crows_pairs_anonymized.csv", header=None)
df_CP.columns = ["","sent_more","sent_less", "stereo_antistereo" ,"bias_type", "annotations", "anon_writer", "anon_annotators"]
df_CP = df_CP.drop(df_CP.columns[0], axis=1)

train_df_cp = df_CP.sample(frac=0.8, random_state=1)
test_df_cp = df_CP.drop(index=train_df_cp.index)
print(len(test_df_cp))
print(len(train_df_cp))
train_df_cp.columns = df_CP.columns
train_df_cp.head()

In [ ]:
import re
import Levenshtein

def preprocess_string(s):
    # 去除非字母字符，转换为小写
    return re.sub(r'[^a-zA-Z]', '', s).lower()

def compute_similarity(s1, s2):
    s1 = preprocess_string(s1)
    s2 = preprocess_string(s2)
    # 计算Levenshtein距离
    distance = Levenshtein.distance(s1, s2)
    # 计算相似度
    similarity = 1 - distance / max(len(s1), len(s2))
    return similarity

str1 = "s"
str2 = "d"

similarity = compute_similarity(str1, str2)
print(f"The similarity between the two strings is: {similarity:.2f}")


In [ ]:
#merged_list = [(item, 'a') for item in a] + [(item, 'b') for item in b]
import random 

def merge_select(list1, list2):
    counter = 0
    merged_list = [(item, 'clean') for item in list1] + [(item, 'poison') for item in list2]
    random.shuffle(merged_list)

    for i in range(len(merged_list)):
        i_s, i_label = merged_list[i]
        i_str = i_s['context']
        for j in range(len(merged_list)-1, i, -1):
            j_s, j_label = merged_list[j]
            j_str = j_s['context']
            s_score = compute_similarity(i_str, j_str)

            if s_score>0.85:
                counter += 1
                del merged_list[j]
        if i >= len(merged_list)-1: break

    clean_p = [item for item, label in merged_list if label == 'clean']
    poison_p = [item for item, label in merged_list if label == 'poison']
    return clean_p, poison_p


def balance(list1, list2, ratio, scaler_num):
    len1 = len(list1)
    len2 = len(list2)
    k_aim = (1-ratio) / ratio
    k_fact = len1 / len2

    if k_fact > k_aim:
        len1_t = int(len2 / ratio * (1-ratio))
        list1_t = random.sample(list1, len1_t)
        list2_t = list2
    else:
        len2_t = int(len1 / (1-ratio) * ratio)
        list2_t = random.sample(list2, len2_t)
        list2_t = list2
    
    if len(list1_t)  > int(scaler_num * (1-ratio)) and len(list2_t)  > int(scaler_num * ratio):
        list1_t = random.sample(list1_t, int(scaler_num * (1-ratio)))
        list2_t = random.sample(list2_t, int(scaler_num * ratio))

    return list1_t, list2_t


In [ ]:
# 主要部分
import os
import json
import random
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument

RAW_KNOWLEDGE_BASE_CP = []

#question: 暂时不用anti-stereo，因为过于中立的没有意义

def processing_cp(attribute='gender', ratio=0.7, scale = 50):

    df_train_attr = train_df_cp[train_df_cp['bias_type'] == attribute]
    df_train_attr = df_train_attr[df_train_attr['stereo_antistereo'] == 'stereo']
    
    list_cp_poison = [{'context': row['sent_more'], 'poison': 'poisoned', 'attr': attribute} \
                      for index, row in df_train_attr.iterrows()]
    list_cp_clean = [{'context': row['sent_less'], 'poison': 'clean', 'attr': attribute} \
                      for index, row in df_train_attr.iterrows()] 
    print(len(list_cp_poison),len(list_cp_clean))
    
    listcp_c, listcp_p = merge_select(list_cp_clean, list_cp_poison)
    print("after",len(listcp_p),len(listcp_c))
    listcp_c, listcp_p = balance(list_cp_clean, list_cp_poison, ratio, scale)
    
    total_text_attribute = []

    for text in listcp_p:
        page_content = text['context'] + '\n'
        metadata = {"source": f"CrowS-Pairs dataset", "attribute": attribute, "poison": "poisoned"}
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE_CP.append(doc)
    for text in listcp_c:
        page_content = text['context'] + '\n'
        metadata = {"source": f"CrowS-Pairs dataset", "attribute": attribute, "poison": "clean"}
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE_CP.append(doc)

    print(f"attribute:{attribute} --- Selected {scale} texts out of {len(df_train_attr)},\
        Clean: {len(listcp_c)}, Poison:{len(listcp_p)}")


ratio = 0.7
scale = 50 
total_texts = []

for attr in stereo_type:
    processing_cp(attr, ratio, scale)
print(f"Created RAW_KNOWLEDGE_BASE with {len(RAW_KNOWLEDGE_BASE_CP)} documents.")
print(f"Example:\n{RAW_KNOWLEDGE_BASE_CP[0]}")

In [ ]:
docs_dict = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE_CP
]

# 将字典列表导出为 JSON 文件
with open(f'CP-{ratio}-{scale}.json', 'w') as json_file:
    json.dump(docs_dict, json_file)

print("Documents exported successfully!")